In [ ]:
!pip install -q transformers datasets evaluate accelerate
!pip install -q matplotlib

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn

from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

print("Torch:", torch.__version__)
import transformers, datasets
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [ ]:
eurosat = load_dataset("eurosat", "rgb")
eurosat

In [ ]:
split_ds = eurosat["train"].train_test_split(test_size=0.2, seed=42)
train_ds = split_ds["train"]
val_ds   = split_ds["test"]

train_ds, val_ds

In [ ]:
label_feature = train_ds.features["label"]
labels = label_feature.names
labels, len(labels)

In [ ]:
def show_examples(dataset_split, n=9):
    idxs = np.random.choice(len(dataset_split), size=n, replace=False)
    cols = 3
    rows = (n + cols - 1) // cols

    plt.figure(figsize=(3*cols, 3*rows))
    for i, idx in enumerate(idxs, 1):
        sample = dataset_split[int(idx)]
        img = sample["image"]
        label = labels[sample["label"]]

        plt.subplot(rows, cols, i)
        plt.imshow(img)
        plt.title(label)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

show_examples(train_ds, n=9)

In [ ]:
model_name = "google/vit-base-patch16-224-in21k"

image_processor = AutoImageProcessor.from_pretrained(model_name)
image_processor

In [ ]:
def transform_examples(batch):
    # batch["image"] — список PIL.Image
    inputs = image_processor(
        images=batch["image"],
        return_tensors="pt"
    )
    batch["pixel_values"] = inputs["pixel_values"]
    return batch

train_ds_transformed = train_ds.with_transform(transform_examples)
val_ds_transformed   = val_ds.with_transform(transform_examples)

In [ ]:
class HFDatasetWrapper(torch.utils.data.Dataset):
    def __init__(self, hf_dataset_split):
        self.ds = hf_dataset_split

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        # EuroSAT: ключи "image", "label", но после with_transform добавился "pixel_values"
        return {
            "pixel_values": item["pixel_values"],
            "labels": item["label"],
        }

train_dataset = HFDatasetWrapper(train_ds_transformed)
eval_dataset  = HFDatasetWrapper(val_ds_transformed)
len(train_dataset), len(eval_dataset)

In [ ]:
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in id2label.items()}

id2label

In [ ]:
num_labels = len(labels)

model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

In [ ]:
#Проверим классификационную голову
model.classifier

In [ ]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels_np = eval_pred
    preds = np.argmax(logits, axis=-1)
    result = accuracy_metric.compute(predictions=preds, references=labels_np)
    return result

In [ ]:
batch_size = 32           # можно уменьшить до 16/8, если не хватает памяти
num_epochs = 5            # для демонстрации
learning_rate = 5e-5

training_args = TrainingArguments(
    output_dir="./vit-eurosat",
    remove_unused_columns=False,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=image_processor,    # формально не обязателен, но ок
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate(eval_dataset=eval_dataset)
print(f"Validation accuracy: {metrics['eval_accuracy']:.4f}")

In [ ]:
raw_example = val_ds[0]
raw_image = raw_example["image"]
true_label = labels[raw_example["label"]]

plt.imshow(raw_image)
plt.title(f"True: {true_label}")
plt.axis("off")
plt.show()

In [ ]:
import torch.nn.functional as F

inputs = image_processor(
    images=raw_image,
    return_tensors="pt"
).to(device)

model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probs = F.softmax(logits, dim=-1)[0]

pred_id = logits.argmax(-1).item()
pred_label = id2label[pred_id]

print(f"True label: {true_label}")
print(f"Predicted:  {pred_label}")

print("\nTop-5 classes:")
top5 = torch.topk(probs, k=5)
for idx, p in zip(top5.indices.tolist(), top5.values.tolist()):
    print(f"{id2label[idx]}: {p:.3f}")